## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/nguyenvanminh281005/bt4222-group12.git"
    BRANCH = "refactor-final"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222 - Group 12")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


Mounted at /content/drive
Code: /content/bt4222_checkout
Features: /content/drive/MyDrive/BT4222 - Group 12/features
Run outputs: /content/drive/MyDrive/BT4222 - Group 12/runs/run_01


# Stacked Recommendation

Combine UBCF, CBF, popularity, availability and prepared user-state signals. Join phase outputs by declared candidate identities before fitting. Use later training-block predictions made by scorers fitted on an earlier block, rather than predictions made on their own fitting observations.

## Meta-models

The logistic stack applies training-fitted scaling and $P(y=1\mid\mathbf x)=\sigma(\mathbf w^\top\mathbf x+b)$. The tree alternative uses LambdaRank when LightGBM is available, with impression groups; otherwise it uses the existing pointwise histogram-gradient-boosting classifier. These alternatives have different objectives and must be identified in results.

Compare feature groups on validation, then evaluate full held-out slates. Training negative sampling and training-based missing-value imputation are explicit. A stronger stack is not evidence that every component helps: use ablation and cold-cohort results to assess contributions and coverage.

Popularity comes exclusively from `4_5_popularity_smoothed.ipynb`, including the earlier-window stack-training export. Run 4_5, 5_1 and 5_2 before this notebook, using the same run directory. Validation/test popularity scores retain the 4_5 update policy; standalone comparisons here use the hybrid evaluator.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run, file_digest

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


In [ ]:
from utils.metrics import evaluate_scores, history_groups


## 1. Configuration and data support


In [ ]:
import json
import time
from functools import lru_cache
import numpy as np
import pandas as pd
from scipy import sparse

OUT_DIR = str(PATHS["outputs"] / "ubcf_cbf")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
DATA_DIR = str(PATHS["raw"])
os.makedirs(CACHE_DIR, exist_ok=True)
SEED = 42
TRAIN_DAYS = ["2019-11-09", "2019-11-10", "2019-11-11", "2019-11-12", "2019-11-13"]
VAL_DAY = "2019-11-14"
TEST_DAY = "2019-11-15"
PHASES = {
    "stack_train": {"build_days": TRAIN_DAYS[:-1], "score_days": [TRAIN_DAYS[-1]]},
    "val": {"build_days": TRAIN_DAYS, "score_days": [VAL_DAY]},
    "test": {"build_days": TRAIN_DAYS, "score_days": [TEST_DAY]},
}
NEG_K = 4
KS = (5, 10)
WARM_MIN_HISTORY = 5
COLD_SEG = "Cold users (0 history)"
SPARSE_SEG = "Sparse users (1-4 history)"
WARM_SEG = "Warm users (>=5 history)"
USER_SEGS = (COLD_SEG, SPARSE_SEG, WARM_SEG)
OUTPUT_COLS = [
    "impression_id",
    "user_id",
    "news_id",
    "label",
    "score",
    "available",
    "user_cold",
    "item_cold",
]


def _has_parquet() -> bool:
    try:
        import pyarrow  # noqa: F401

        return True
    except ImportError:
        return False


def save_df(df: pd.DataFrame, path_no_ext: str) -> str:
    if _has_parquet():
        path = path_no_ext + ".parquet"
        df.to_parquet(path, index=False)
    else:
        path = path_no_ext + ".pkl"
        df.to_pickle(path)
    return path


def load_df(path_no_ext: str) -> pd.DataFrame:
    if os.path.exists(path_no_ext + ".parquet"):
        return pd.read_parquet(path_no_ext + ".parquet")
    return pd.read_pickle(path_no_ext + ".pkl")


def load_behaviors():
    beh, _ = load_preprocessed(PATHS)
    beh["history"] = beh["history"].str.split()
    beh["date"] = beh["time"].dt.strftime("%Y-%m-%d")
    return beh


def load_news():
    return pd.read_pickle(PATHS["features"] / "hybrid_news.pkl")


In [ ]:
@lru_cache(maxsize=1)
def user_history_groups() -> pd.Series:
    """impression_id -> reporting group, from the length of the impression's raw MIND `history`
    (same definition as the MLP's cold_user / sparse_user / warm_user slices).
    Evaluation only: unlike user_cold it ignores clicks in the build window, so a user
    in the cold group can still have build-window clicks for the base models to use."""
    n = load_behaviors().set_index("impression_id")["history"].str.len()
    return pd.Series(
        np.select([n == 0, n < WARM_MIN_HISTORY], [COLD_SEG, SPARSE_SEG], WARM_SEG),
        index=n.index,
    )


In [ ]:
def impute_train_mean(train_df: pd.DataFrame, dfs: list, cols) -> dict:
    """Fill NaNs in `cols` of every frame in `dfs` (in place) with the TRAINING-set mean.
    Pair every imputed column with an availability flag so the model can tell
    'unknown' from 'average'. Returns the means used."""
    means = {
        c: float(train_df[c].mean()) if train_df[c].notna().any() else 0.0 for c in cols
    }
    for d in dfs:
        for c in cols:
            d[c] = d[c].fillna(means[c])
    return means


def sample_negatives(
    df: pd.DataFrame, k: int = NEG_K, seed: int = SEED
) -> pd.DataFrame:
    """Group policy: keep every positive, sample up to k negatives per positive within the same
    impression. Applied to model TRAINING rows only - evaluation always uses full slates.
    """
    rng = np.random.default_rng(seed)
    d = df.copy()
    d["_r"] = rng.random(len(d))
    npos = d.groupby("impression_id")["label"].transform("sum")
    neg = d["label"] == 0
    d["_nr"] = np.inf
    d.loc[neg, "_nr"] = d[neg].groupby("impression_id")["_r"].rank(method="first")
    keep = (d["label"] == 1) | (d["_nr"] <= k * npos)
    return d[keep & (npos > 0)].drop(columns=["_r", "_nr"]).reset_index(drop=True)


In [ ]:
def evaluate(df, score_col="score", ks=KS, seed=SEED, by_segment=True):
    return evaluate_scores(df, score_col, ks, seed, by_segment, user_history_groups())


In [ ]:
def save_scores(df: pd.DataFrame, model: str, phase: str, extra_cols=()) -> str:
    """Write a model's scores in the shared format (+ optional model-specific feature cols)."""
    missing = [c for c in OUTPUT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"missing output columns: {missing}")
    cols = OUTPUT_COLS + [c for c in extra_cols if c not in OUTPUT_COLS]
    out = df[cols].copy()
    out["available"] = out["available"].astype(np.int8)
    os.makedirs(OUT_DIR, exist_ok=True)
    return save_df(out, os.path.join(OUT_DIR, f"scores_{model}_{phase}"))


def load_popularity_scores(phase):
    """Load the canonical 4_5 scores; validate their dataset, identity and coverage."""
    directory = PATHS["outputs"] / "base_models" / "popularity_smoothed"
    manifest_path = directory / "stacking_scores.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("Run 4_5_popularity_smoothed.ipynb through its final stacking export cell first.")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    preprocessing = json.loads(
        (PATHS["processed"] / "preprocessing" / "metadata.json").read_text()
    )
    if manifest["schema_version"] != 1 or manifest["source_files"] != preprocessing["source_files"]:
        raise ValueError("Popularity scores belong to different source data. Rerun 4_5.")
    artifact = manifest["files"][phase]
    path = directory / artifact["name"]
    if file_digest(path) != artifact["sha256"]:
        raise ValueError("Popularity score contents changed. Rerun 4_5.")
    frame = load_table(directory / f"{phase}_scores")
    frame = frame.merge(
        beh[["impression_id", "user_id", "history"]],
        on="impression_id", how="left", validate="many_to_one",
    )
    assert frame["user_id"].notna().all() and frame["score"].notna().all()
    frame["user_cold"] = frame["history"].map(len).eq(0).astype(np.int8)
    frame["item_cold"] = frame["is_cold_article"].astype(np.int8)
    frame["available"] = np.int8(1)
    frame["pop_ctr"] = frame["score"]
    frame["pop_log_exps"] = frame["log_exposure"]
    return frame[OUTPUT_COLS + ["pop_ctr", "pop_log_exps"]]


def load_scores(model: str, phase: str) -> pd.DataFrame:
    if model == "pop":
        return load_popularity_scores(phase)
    return load_df(os.path.join(OUT_DIR, f"scores_{model}_{phase}"))


def standalone_eval(model: str, score_col: str = "score") -> dict:
    """Evaluate a base model on its own: unavailable scores -> training-set mean
    (mean of available scores in the stack_train phase, i.e. scored train-period day).
    """
    tr = load_scores(model, "stack_train")
    fill = (
        float(tr.loc[tr["available"] == 1, score_col].mean())
        if (tr["available"] == 1).any()
        else 0.0
    )
    res = {}
    for ph in ("val", "test"):
        d = load_scores(model, ph)
        d[score_col] = d[score_col].where(d["available"] == 1, fill).fillna(fill)
        res[ph] = evaluate(d, score_col)
    return res


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

beh = load_behaviors()
news = load_news()

try:
    import lightgbm as lgb

    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print(
        "lightgbm not installed -> using sklearn HistGradientBoosting (pointwise) instead of LambdaRank"
    )


## 2. User-state features (per impression, information before t only)


In [ ]:
ustate = pd.read_pickle(PATHS["features"] / "hybrid_user_state.pkl")


## 3. Assemble one frame per phase


In [ ]:
KEYS = ["impression_id", "news_id"]


def load_phase(phase):
    u = load_scores("ubcf", phase).rename(
        columns={"score": "ubcf_score", "available": "ubcf_available"}
    )
    c = load_scores("cbf", phase).rename(
        columns={"score": "cbf_score", "available": "cbf_available"}
    )
    p = load_scores("pop", phase).rename(columns={"available": "pop_available"})
    base = ["user_id", "label", "user_cold", "item_cold"]
    for name, d in (("ubcf", u), ("cbf", c), ("pop", p)):
        if d.duplicated(KEYS).any():
            print(
                f"WARNING: duplicate (impression_id, news_id) in {name}/{phase} - keeping first"
            )
    u, c, p = (d.drop_duplicates(KEYS) for d in (u, c, p))
    df = (
        u.merge(c.drop(columns=base), on=KEYS, how="inner")
        .merge(p.drop(columns=base + ["score"]), on=KEYS, how="inner")
        .merge(ustate, on="impression_id", how="left")
    )
    assert len(df) == len(u), f"{phase}: row mismatch after merge"
    return df


data = {ph: load_phase(ph) for ph in PHASES}
{
    ph: f"{len(d):,} rows / {d['impression_id'].nunique():,} impressions"
    for ph, d in data.items()
}


{'stack_train': '1,231,673 rows / 31,624 impressions',
 'val': '1,222,429 rows / 30,270 impressions',
 'test': '2,740,998 rows / 73,152 impressions'}

## 4. Features
Group **A** collaborative, **B** content, **C** popularity, **D** user state. Each imputed signal has its availability flag next to it so "unknown" ≠ "average".

`pop_log_exps` (prior exposure count) is **off by default**: it partly acts as a recency proxy, and `article_recency` is banned by the contract – agree with the group before switching it on.


In [ ]:
USE_EXPOSURE_COUNT = False
GROUPS = {
    "A_ubcf": ["ubcf_score", "ubcf_available", "ubcf_strength", "ubcf_n_clicks"],
    "B_cbf": ["cbf_content", "cbf_kg", "cbf_cat_aff", "cbf_available"],
    "C_pop": ["pop_ctr", "pop_available", "item_cold"]
    + (["pop_log_exps"] if USE_EXPOSURE_COUNT else []),
    "D_user": [
        "user_cold",
        "log_history_length",
        "reading_entropy",
        "hour_sin",
        "hour_cos",
    ],
}
FEATURES = [f for g in GROUPS.values() for f in g]
# contract: missing values -> mean of the (stacker's) training set
means = impute_train_mean(data["stack_train"], list(data.values()), FEATURES)
train = sample_negatives(data["stack_train"], k=NEG_K)
print(
    f"stacker training rows after negative sampling: {len(train):,} "
    f"(positives {train['label'].mean():.1%})"
)


stacker training rows after negative sampling: 230,137 (positives 20.8%)


## 5. Shared stacker functions
Define LR, GBM, and prediction helpers before training either feature set. LightGBM uses LambdaRank with validation early stopping; otherwise GBM uses the scikit-learn classifier fallback.


In [ ]:
def fit_lr(feats):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    m.fit(train[feats], train["label"])
    return m


def fit_gbm(feats):
    if HAS_LGB:
        tr = train.sort_values("impression_id")
        va = data["val"].sort_values("impression_id")
        m = lgb.LGBMRanker(
            objective="lambdarank",
            n_estimators=2000,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=50,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=SEED,
            verbose=-1,
        )
        m.fit(
            tr[feats],
            tr["label"],
            group=tr.groupby("impression_id", sort=False).size().to_numpy(),
            eval_set=[(va[feats], va["label"])],
            eval_group=[va.groupby("impression_id", sort=False).size().to_numpy()],
            eval_at=[5, 10],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        return m
    from sklearn.ensemble import HistGradientBoostingClassifier

    m = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, early_stopping=True, random_state=SEED
    )
    return m.fit(train[feats], train["label"])


def predict(m, df, feats):
    if HAS_LGB and isinstance(m, lgb.LGBMRanker):
        return m.predict(df[feats])
    return m.predict_proba(df[feats])[:, 1]


## 6. Stage 1 · Train without popularity
Train LR and GBM using UBCF, CBF, and user-state features. Exclude the entire `C_pop` group. Both stages use the same sampled training rows and evaluation splits.


In [ ]:
# Keep collaborative, content, and user-state features; exclude the entire popularity group.
FEATURES_NO_POP = [
    f for group, feats in GROUPS.items() if group != "C_pop" for f in feats
]
lr_no_pop = fit_lr(FEATURES_NO_POP)
gbm_no_pop = fit_gbm(FEATURES_NO_POP)
no_pop_results = {}

for name, model, score_name in (
    ("Hybrid-LR-no-pop", lr_no_pop, "hybrid_lr_no_pop"),
    ("Hybrid-GBM-no-pop", gbm_no_pop, "hybrid_gbm_no_pop"),
):
    for phase in ("val", "test"):
        scored = data[phase].assign(
            score=predict(model, data[phase], FEATURES_NO_POP), available=1
        )
        save_scores(scored, score_name, phase)
        metrics = evaluate(scored)[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
        no_pop_results[(name, phase)] = metrics
        metrics.to_csv(f"{OUT_DIR}/metrics_{score_name}_{phase}.csv")
        print(f"\n=== {name} · {phase} ===")
        print(metrics.round(4).to_string())



=== Hybrid-LR-no-pop · val ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          30270  0.3507  0.3265   0.3842  0.6466
Cold users (0 history)                         772  0.2957  0.2753   0.3307  0.5849
Sparse users (1-4 history)                    3247  0.3258  0.3151   0.3767  0.6045
Warm users (>=5 history)                     26251  0.3554  0.3294   0.3867  0.6537
Clicked article seen before                   8276  0.2319  0.2106   0.2642  0.5525
Clicked article new (no prior clicks)        21994  0.3955  0.3701   0.4294  0.6821

=== Hybrid-LR-no-pop · test ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          73152  0.3622  0.3456   0.4040  0.6411
Cold users (0 history)                        2214  0.2629  0.2428   0.3016  0.5093
Sparse users (1-4 history)                    8092  0.3527  0.3554   0.4102  0.5939
Warm users 

## 7. Stage 2 · Add popularity and train full stacks
Add the `C_pop` feature group and fit fresh LR and GBM models for comparison. These are separate fits, rather than continued training of the stage 1 models.


In [ ]:
lr = fit_lr(FEATURES)
coef = pd.Series(lr[-1].coef_[0], index=FEATURES).sort_values(
    key=np.abs, ascending=False
)
print("LR standardised coefficients:\n", coef.round(3).to_string())
gbm = fit_gbm(FEATURES)

if HAS_LGB:
    imp = pd.Series(gbm.booster_.feature_importance("gain"), index=FEATURES)
    print("best iteration:", gbm.best_iteration_)
    print(
        "gain importance:\n",
        (imp / imp.sum()).sort_values(ascending=False).round(3).to_string(),
    )


LR standardised coefficients:
 cbf_content           0.467
pop_ctr               0.249
log_history_length   -0.229
cbf_cat_aff           0.154
ubcf_available       -0.089
reading_entropy       0.088
item_cold             0.088
ubcf_strength         0.056
hour_cos             -0.033
cbf_kg                0.026
ubcf_score            0.022
ubcf_n_clicks         0.013
user_cold            -0.012
cbf_available         0.012
hour_sin              0.011
pop_available         0.000
best iteration: 141
gain importance:
 cbf_content           0.392
pop_ctr               0.257
cbf_cat_aff           0.095
cbf_kg                0.080
log_history_length    0.034
hour_sin              0.032
ubcf_score            0.023
ubcf_n_clicks         0.021
reading_entropy       0.020
ubcf_strength         0.020
hour_cos              0.019
ubcf_available        0.004
item_cold             0.003
cbf_available         0.000
user_cold             0.000
pop_available         0.000


## 8. Ablation on validation – what does each feature group add?


In [ ]:
abl = []

for drop in [None] + list(GROUPS):
    feats = [f for f in FEATURES if drop is None or f not in GROUPS[drop]]
    m = fit_gbm(feats)
    r = evaluate(data["val"].assign(score=predict(m, data["val"], feats)))
    abl.append(
        {
            "dropped": drop or "(none)",
            **{f"{seg} nDCG@10": r.loc[seg, "nDCG@10"] for seg in r.index},
        }
    )

ablation = pd.DataFrame(abl).set_index("dropped")
ablation.round(4)


,All nDCG@10,Cold users (0 history) nDCG@10,Sparse users (1-4 history) nDCG@10,Warm users (>=5 history) nDCG@10,Clicked article seen before nDCG@10,Clicked article new (no prior clicks) nDCG@10
dropped,,,,,,
(none),0.3943,0.3413,0.3867,0.3968,0.3449,0.4129
A_ubcf,0.3934,0.3369,0.3865,0.3959,0.3425,0.4125
B_cbf,0.3019,0.3018,0.3268,0.2989,0.3722,0.2755
C_pop,0.3843,0.3247,0.3781,0.3868,0.3015,0.4154
D_user,0.3937,0.3339,0.3861,0.3964,0.3443,0.4123


## 9. Final comparison: base models vs stacked hybrid (val & test, by segment)


In [ ]:
results = dict(no_pop_results)

for model in ("ubcf", "cbf", "pop"):
    for ph, t in standalone_eval(model).items():
        results[(model.upper(), ph)] = t

for name, m in (("Hybrid-LR", lr), ("Hybrid-GBM", gbm)):
    for ph in ("val", "test"):
        d = data[ph].assign(score=predict(m, data[ph], FEATURES))
        results[(name, ph)] = evaluate(d)
        if name == "Hybrid-GBM":
            d["available"] = 1
            save_scores(d, "hybrid", ph)

summary = pd.concat(
    {k: v for k, v in results.items()}, names=["model", "phase", "segment"]
)
summary = summary[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
summary.to_csv(f"{OUT_DIR}/metrics_all_models.csv")
ORDER = [
    "UBCF",
    "CBF",
    "POP",
    "Hybrid-LR-no-pop",
    "Hybrid-GBM-no-pop",
    "Hybrid-LR",
    "Hybrid-GBM",
]
SEGS = list(results[("UBCF", "val")].index)

for ph in ("val", "test"):
    print(f"\n===== {ph.upper()} =====")
    s = summary.xs(ph, level="phase")
    print("\nAll impressions:")
    print(s.xs("All", level="segment").reindex(ORDER).round(4).to_string())
    print("\nnDCG@10 by segment:")
    print(
        s["nDCG@10"]
        .unstack("segment")
        .reindex(index=ORDER, columns=SEGS)
        .round(4)
        .to_string()
    )



===== VAL =====

All impressions:
                   impressions     MRR  nDCG@5  nDCG@10     AUC
model                                                          
UBCF                     30270  0.2546  0.2276   0.2866  0.5304
CBF                      30270  0.3483  0.3242   0.3830  0.6468
POP                      30270  0.2670  0.2415   0.3037  0.5741
Hybrid-LR-no-pop         30270  0.3507  0.3265   0.3842  0.6466
Hybrid-GBM-no-pop        30270  0.3515  0.3266   0.3843  0.6458
Hybrid-LR                30270  0.3595  0.3344   0.3939  0.6630
Hybrid-GBM               30270  0.3613  0.3353   0.3943  0.6575

nDCG@10 by segment:
segment               All  Cold users (0 history)  Sparse users (1-4 history)  Warm users (>=5 history)  Clicked article seen before  Clicked article new (no prior clicks)
model                                                                                                                                                                      
UBCF               0.286

## 10. Performance by user group
Groups use the length of the impression's raw `history`, the same definition as the MLP slices: **cold** = 0, **sparse** = 1–4, **warm** = ≥5. Every model is ranked on the same impressions, on full slates.


In [ ]:
USER_GROUPS = ["All", *USER_SEGS]
by_user = summary[summary.index.get_level_values("segment").isin(USER_GROUPS)]
by_user.to_csv(f"{OUT_DIR}/metrics_user_groups.csv")

for ph in ("val", "test"):
    print(f"\n===== {ph.upper()} =====")
    for seg in USER_GROUPS:
        s = by_user.xs((ph, seg), level=("phase", "segment")).reindex(ORDER)
        print(f"\n{seg} - {int(s['impressions'].iloc[0]):,} impressions")
        print(s.drop(columns="impressions").round(4).to_string())



===== VAL =====

All - 30,270 impressions
                      MRR  nDCG@5  nDCG@10     AUC
model                                             
UBCF               0.2546  0.2276   0.2866  0.5304
CBF                0.3483  0.3242   0.3830  0.6468
POP                0.2670  0.2415   0.3037  0.5741
Hybrid-LR-no-pop   0.3507  0.3265   0.3842  0.6466
Hybrid-GBM-no-pop  0.3515  0.3266   0.3843  0.6458
Hybrid-LR          0.3595  0.3344   0.3939  0.6630
Hybrid-GBM         0.3613  0.3353   0.3943  0.6575

Cold users (0 history) - 772 impressions
                      MRR  nDCG@5  nDCG@10     AUC
model                                             
UBCF               0.2572  0.2347   0.2841  0.5261
CBF                0.2918  0.2706   0.3300  0.5860
POP                0.2693  0.2468   0.3074  0.5751
Hybrid-LR-no-pop   0.2957  0.2753   0.3307  0.5849
Hybrid-GBM-no-pop  0.2853  0.2646   0.3247  0.5817
Hybrid-LR          0.3059  0.2816   0.3414  0.6039
Hybrid-GBM         0.3032  0.2766   0.3413  0.60